# Feature Extraction

Extract image descriptors or other features for model training.

In [11]:
print('Feature extraction notebook placeholder')

Feature extraction notebook placeholder


In [12]:
import os
import tarfile
import shutil

tar_path = "CUB_200_2011.tgz"
extract_dir = "./raw_cub"
dataset_url = "https://data.caltech.edu/records/65de6-vp158/files/CUB_200_2011.tgz"
dataset_dir = "./dataset_20_species"

if not os.path.exists(dataset_dir):
    if not os.path.exists(tar_path):
        print("Downloading CUB-200 dataset...")
        !wget --no-check-certificate {dataset_url} -O {tar_path}

    if not os.path.exists(extract_dir):
        print("Extracting files...")
        with tarfile.open(tar_path, "r:gz") as tar:
            tar.extractall(path=extract_dir)

    source_images_path = os.path.join(extract_dir, "CUB_200_2011", "images")
    os.makedirs(dataset_dir, exist_ok=True)
    all_available_species = sorted(os.listdir(source_images_path))

    bd_keywords = [
        'Crow', 'Kingfisher', 'Hummingbird', 'Mallard', 'Warbler',
        'Towhee', 'Jay', 'Creeper', 'Waxwing', 'Cuckoo',
        'Thrush', 'Woodpecker', 'Wren', 'Vireo', 'Catbird',
        'Meadowlark', 'Blackbird', 'Gull', 'Tern', 'Pelican'
    ]

    selected_species = []
    for species_folder in all_available_species:
        for kw in bd_keywords:
            if kw.lower() in species_folder.lower():
                if species_folder not in selected_species:
                    selected_species.append(species_folder)
                break
        if len(selected_species) == 20:
            break

    for species in selected_species:
        src = os.path.join(source_images_path, species)
        dst = os.path.join(dataset_dir, species)
        shutil.copytree(src, dst)

    print(f"Dataset ready at '{dataset_dir}'!")
else:
    print(f"Dataset already exists at '{dataset_dir}'!")

Dataset already exists at './dataset_20_species'!


In [13]:
import cv2
import numpy as np
import joblib
from pathlib import Path
from skimage.feature import hog

OUTPUT_DIR = Path("./processed_data")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# HOG Parameters
IMG_SIZE = (128, 128)
ORIENTATIONS = 9
PIXELS_PER_CELL = (8, 8)
CELLS_PER_BLOCK = (2, 2)

X_hog = []
y_labels = []

classes = sorted([d for d in os.listdir(dataset_dir) if os.path.isdir(os.path.join(dataset_dir, d))])
label_mapping = {class_name: idx for idx, class_name in enumerate(classes)}

print(f"Extracting HOG features across {len(classes)} bird species...")

for class_name in classes:
    class_path = os.path.join(dataset_dir, class_name)
    label = label_mapping[class_name]

    for img_name in os.listdir(class_path):
        if img_name.lower().endswith(('.jpg', '.jpeg', '.png')):
            img_path = os.path.join(class_path, img_name)
            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            if img is None:
                continue

            img_resized = cv2.resize(img, IMG_SIZE)
            hog_vector = hog(
                img_resized,
                orientations=ORIENTATIONS,
                pixels_per_cell=PIXELS_PER_CELL,
                cells_per_block=CELLS_PER_BLOCK,
                block_norm='L2-Hys',
                visualize=False
            )

            X_hog.append(hog_vector)
            y_labels.append(label)

X_hog = np.array(X_hog, dtype=np.float32)
y_labels = np.array(y_labels, dtype=np.int64)

print("\nHOG Feature Extraction Complete!")
print(f"HOG Matrix Shape (X): {X_hog.shape}")
print(f"Labels Shape (y)     : {y_labels.shape}")

Extracting HOG features across 20 bird species...

HOG Feature Extraction Complete!
HOG Matrix Shape (X): (1168, 8100)
Labels Shape (y)     : (1168,)


In [14]:
# Save feature matrices for Week 4 machine learning algorithms
np.save(OUTPUT_DIR / "X_hog.npy", X_hog)
np.save(OUTPUT_DIR / "y_labels.npy", y_labels)
joblib.dump(label_mapping, OUTPUT_DIR / "label_mapping.pkl")

print(f"Saved feature vectors to '{OUTPUT_DIR}' successfully!")
print("Files generated:")
print("  - processed_data/X_hog.npy")
print("  - processed_data/y_labels.npy")
print("  - processed_data/label_mapping.pkl")

Saved feature vectors to 'processed_data' successfully!
Files generated:
  - processed_data/X_hog.npy
  - processed_data/y_labels.npy
  - processed_data/label_mapping.pkl


In [15]:
import numpy as np
import joblib

# 1. Load the files
X_hog = np.load("./processed_data/X_hog.npy")
y_labels = np.load("./processed_data/y_labels.npy")
label_mapping = joblib.load("./processed_data/label_mapping.pkl")

# 2. Inspect HOG Features (X_hog)
print("--- HOG Feature Matrix (X_hog) ---")
print(f"Shape: {X_hog.shape}")
print(f"Sample features for 1st image (first 10 numbers):")
print(X_hog[0][:10])  # Prints the first 10 gradient values

print("\n--- Target Labels (y_labels) ---")
print(f"Shape: {y_labels.shape}")
print(f"First 15 image labels: {y_labels[:15]}")

print("\n--- Species Label Mapping (label_mapping.pkl) ---")
for species_name, class_id in list(label_mapping.items())[:5]:
    print(f"Class ID {class_id} -> {species_name}")

--- HOG Feature Matrix (X_hog) ---
Shape: (1168, 8100)
Sample features for 1st image (first 10 numbers):
[0.3062033  0.2430128  0.12994528 0.05496656 0.16903019 0.04581124
 0.06881208 0.0323204  0.08324099 0.3157302 ]

--- Target Labels (y_labels) ---
Shape: (1168,)
First 15 image labels: [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]

--- Species Label Mapping (label_mapping.pkl) ---
Class ID 0 -> 009.Brewer_Blackbird
Class ID 1 -> 010.Red_winged_Blackbird
Class ID 2 -> 011.Rusty_Blackbird
Class ID 3 -> 012.Yellow_headed_Blackbird
Class ID 4 -> 018.Spotted_Catbird
